# NumPy Arrays and Data Types — Runnable Examples

This notebook demonstrates array dimensions, dtype selection, type conversion, finite/missing values, memory use, and views versus copies.

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.3.5


## 1. Array dimensions and attributes

In [2]:
scalar = np.array(7)
vector = np.array([1, 2, 3])
matrix = np.arange(12).reshape(3, 4)
cube = np.zeros((2, 3, 4))
for name, arr in [("scalar", scalar), ("vector", vector), ("matrix", matrix), ("cube", cube)]:
    print(name, "shape=", arr.shape, "ndim=", arr.ndim, "size=", arr.size, "dtype=", arr.dtype)

scalar shape= () ndim= 0 size= 1 dtype= int64
vector shape= (3,) ndim= 1 size= 3 dtype= int64
matrix shape= (3, 4) ndim= 2 size= 12 dtype= int64
cube shape= (2, 3, 4) ndim= 3 size= 24 dtype= float64


## 2. Explicit dtypes and integer limits

In [3]:
signed = np.array([-128, 0, 127], dtype=np.int8)
unsigned = np.array([0, 128, 255], dtype=np.uint8)
floats = np.array([1.25, 2.5], dtype=np.float32)
print(signed, signed.dtype, signed.itemsize)
print(unsigned, unsigned.dtype, unsigned.itemsize)
print(floats, floats.dtype, floats.itemsize)
iinfo = np.iinfo(np.int8)
uinfo = np.iinfo(np.uint8)
print("int8 range:", iinfo.min, "to", iinfo.max)
print("uint8 range:", uinfo.min, "to", uinfo.max)

[-128    0  127] int8 1
[  0 128 255] uint8 1
[1.25 2.5 ] float32 4
int8 range: -128 to 127
uint8 range: 0 to 255


## 3. Type inference and casting

In [4]:
mixed = np.array([1, 2.5, 3])
print("inferred:", mixed, mixed.dtype)
values = np.array([1.2, 2.8, -3.7])
print("truncated:", values.astype(np.int64))
print("rounded then cast:", np.rint(values).astype(np.int64))
converted = np.array([1, 2, 3], dtype=np.int32).astype(np.float64)
print("converted dtype:", converted.dtype)
print("shares memory with source:", np.shares_memory(values, values.astype(int)))

inferred: [1.  2.5 3. ] float64
truncated: [ 1  2 -3]
rounded then cast: [ 1  3 -4]
converted dtype: float64
shares memory with source: False


## 4. Shape operations versus dtype conversion

In [5]:
a = np.arange(12, dtype=np.float32)
reshaped = a.reshape(3, 4)
converted = a.astype(np.float64)
print("original shape/dtype:", a.shape, a.dtype)
print("reshaped shape/dtype:", reshaped.shape, reshaped.dtype)
print("converted shape/dtype:", converted.shape, converted.dtype)

original shape/dtype: (12,) float32
reshaped shape/dtype: (3, 4) float32
converted shape/dtype: (12,) float64


## 5. NaN, infinity, and finite values

In [6]:
x = np.array([1.0, np.nan, np.inf, -np.inf, 5.0])
print("is NaN:", np.isnan(x))
print("is finite:", np.isfinite(x))
print("finite values:", x[np.isfinite(x)])

is NaN: [False  True False False False]
is finite: [ True False False False  True]
finite values: [1. 5.]


## 6. Memory usage

In [7]:
a32 = np.zeros(100, dtype=np.float32)
a64 = np.zeros(100, dtype=np.float64)
print("float32 bytes:", a32.nbytes)
print("float64 bytes:", a64.nbytes)
assert a32.nbytes == 400
assert a64.nbytes == 800

float32 bytes: 400
float64 bytes: 800


## 7. Views and copies

In [8]:
original = np.arange(6)
view = original[1:4]
print("shares memory (view):", np.shares_memory(original, view))
view[0] = 99
print("original after view mutation:", original)
original2 = np.arange(6)
copy = original2[1:4].copy()
print("shares memory (copy):", np.shares_memory(original2, copy))
copy[0] = 99
print("original after copy mutation:", original2)

shares memory (view): True
original after view mutation: [ 0 99  2  3  4  5]
shares memory (copy): False
original after copy mutation: [0 1 2 3 4 5]


## 8. Validate a dtype range before casting

In [9]:
def safe_uint8(values):
    arr = np.asarray(values)
    if not np.issubdtype(arr.dtype, np.number):
        raise TypeError("Numeric values are required.")
    if not np.isfinite(arr).all():
        raise ValueError("Values must all be finite.")
    if (arr < 0).any() or (arr > 255).any():
        raise ValueError("Every value must be in the range 0 to 255.")
    if not np.equal(arr, np.floor(arr)).all():
        raise ValueError("Values must be integers.")
    return arr.astype(np.uint8)

print(safe_uint8([0, 128, 255]))
try:
    safe_uint8([0, 300])
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)

[  0 128 255]
ValueError: Every value must be in the range 0 to 255.


## 9. Mini task: inspect and filter a dataset

In [10]:
data = np.array([[18, 70], [21, 82], [19, 65], [25, 91]], dtype=np.float32)
print("shape:", data.shape, "dtype:", data.dtype)
print("rows with score >= 70:\n", data[data[:, 1] >= 70])
print("column means:", data.mean(axis=0))
assert data.shape == (4, 2)
assert np.isfinite(data).all()
print("All checks passed.")

shape: (4, 2) dtype: float32
rows with score >= 70:
 [[18. 70.]
 [21. 82.]
 [25. 91.]]
column means: [20.75 77.  ]
All checks passed.
